# Step 6: reproduce Akula's cache-v2 result, then our ViT-only model on cache v2 (GPU, ~3 h)
Akula's update: B3 (ViT image-only, `b3_vit`, seed 0) on cache v2 gave **val AUC 0.559** (old cache: 0.530).
1. **B3 reproduction:** the same procedure (`b3_vit`, seed 0, cache v2, no overrides) and a paired
   comparison with his committed predictions. GPU training is not bit-exact; an AUC within ~0.03 and highly
   correlated predictions count as a match.
2. **Probe on cache v2:** our step-4 sweep again. The first probe used the old clipped cache; v2 has a
   tumour-based clip ceiling and a fixed mm scale.
3. **Our ViT-only model on cache v2:** the probe's top setting, 3 seeds, ensemble, then paired DeLong and
   bootstrap tests against B3.

Kaggle setup: Accelerator **GPU T4 x2** (one GPU is used), Internet **ON**. **+ Add Input**:
- **`mnj-cache-v2`** (Akula's builder-v2 cache, all 3 cohorts). With `CACHE_VERSION = 2` the run stops if
  another cache is picked.
- the code: the `mnj-code-vaib` Dataset (upload the new zip as a **new version** first) or the `GITHUB_TOKEN` secret.

Use **Save Version -> Save & Run All (Commit)**: these runs take hours and continue with the browser closed.
Results go to the Output tab under `results/<PERSON>/<step>/`. Keep this notebook's output: step 8 reads it.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "vaib"           # branch (or tag) to clone
PERSON = ""               # optional: "A", "B" or "C" -> results/<PERSON>/ (empty -> results/vaib/)
STEP = "step6"
CACHE_VERSION = 2          # Akula's mnj-cache-v2
RUN_B3 = True              # 1. reproduce Akula's B3 (~0.5 h)
RUN_PROBE = True           # 2. probe on cache v2 (~0.5 h)
PROBE_BACKBONES = ["mae", "dinov2_b"]
PROBE_NORMS = ["joint_pct", "per_phase", "joint_tumor", "pre_ref", "zscore", "sub", "kinetic", "early3"]
PROBE_CROPS = ["fixed", "tumor"]
RUN_VITONLY = True         # 3. fine-tune the probe's top setting (~0.5 h per seed)
VITONLY_SEEDS = [0, 1, 2]
VITONLY_EXTRA = []         # extra overrides, e.g. ["model.train_blocks=2"]
TIME_BUDGET_H = 11.0
DEBUG = False              # True: 2 epochs on 48 patients per run (a few minutes) to check the notebook
PUSH_RESULTS = False
NOTEBOOK_NAME = "step6"

In [ ]:
# ==== SETUP (do not edit): get the repo code, install requirements ====
import base64, json, os, shutil, subprocess, sys
if PERSON not in ("A", "B", "C"):           # only names the results folder
    PERSON = "vaib"
    print('PERSON not set: results go to results/vaib/ (set PERSON = "A", "B" or "C" to change)')
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:
    print("no GITHUB_TOKEN secret attached to this notebook (fine if the mnj-code-vaib Dataset is attached):",
          type(e).__name__)

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    if not _tok:
        raise RuntimeError("no GITHUB_TOKEN secret: git push/pull is not available")
    hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.extraHeader={hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
if _tok:                                    # 1) clone the branch from GitHub
    git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
    print("code: GitHub", REPO, GIT_TAG, git("log", "-1", "--format=%h %s"))
else:                                       # 2) or a code Dataset (zip of the repo) attached as input
    import glob, zipfile
    hits = [h for d in range(1, 8) for h in glob.glob("/kaggle/input/" + "*/" * d + "src/vitonly/__init__.py")]
    zips = [z for d in range(1, 6) for z in glob.glob("/kaggle/input/" + "*/" * d + "mnj-code-vaib*.zip")]
    if not hits and zips:                   # Kaggle kept the uploaded zip as a file: unpack it
        zipfile.ZipFile(zips[0]).extractall("/tmp/code_zip")
        hits = glob.glob("/tmp/code_zip/**/src/vitonly/__init__.py", recursive=True)
    if not hits:
        print("attached inputs:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if root.count("/") <= 5:
                print("  ", root, dirs[:6], files[:4])
            else:
                dirs[:] = []
        raise SystemExit(
            "\nThe notebook can't find the project code. Do ONE of these, then Run All again:\n"
            "  (a) Add-ons -> Secrets -> add GITHUB_TOKEN (a GitHub token with access to the repo) and switch it ON\n"
            "      for this notebook, or\n"
            "  (b) + Add Input -> Datasets -> Your Datasets -> mnj-code-vaib (made by uploading mnj-code-vaib.zip).")
    code_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    shutil.copytree(code_root, "/tmp/repo")
    print("code: attached Dataset", code_root)
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")         # so `from src...` works in this kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ==== HELPERS (do not edit): training runs, prediction files, comparisons ====
import glob, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import evaluation as ev, stats
from src.train import find_caches
from src.data.metadata import patient_table
COH = ["ispy1", "ispy2", "duke"]
OUT = f"/kaggle/working/results/{PERSON}/{STEP}"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
RUNS_JSON = {}
pd.set_option("display.width", 250, "display.max_columns", 40)

def budget_left():
    return TIME_BUDGET_H - (time.time() - T0) / 3600

def check_caches(version):
    caches = find_caches(["/kaggle/input", "/kaggle/working"])
    info = {c: json.load(open(f"{d}/cache_meta.json")).get("builder_version", 1) for c, d in caches.items()}
    print("caches:", {c: f"builder v{v}: {caches[c]}" for c, v in info.items()})
    if version is not None:
        bad = {c: v for c, v in info.items() if v != version}
        assert not bad and len(info) == 3, (f"need cache builder v{version} for all 3 cohorts (attach mnj-cache-v2); "
                                            f"found {info}")
    return caches

def _copy(d, label):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    for f in ("val_preds.csv", "metrics.json", "log.csv", "config_resolved.yaml", "export_preds.csv"):
        if os.path.exists(f"{d}/{f}"):
            shutil.copy(f"{d}/{f}", f"{OUT}/{label}/{f}")

def train(config, seed, sets=(), label=None, cohorts=None):
    """One src.train run (B3 / F2 ...) -> (run_dir, metrics). Its small files are copied to OUT/<label>/."""
    cmd = [sys.executable, "-m", "src.train", "--config", config, "--seed", str(seed), "--out", "/kaggle/working/runs"]
    if CACHE_VERSION is not None: cmd += ["--cache-version", str(CACHE_VERSION)]
    if cohorts: cmd += ["--cohorts", *cohorts]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    print("\n" + " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d, json.load(open(f"{d}/metrics.json"))

def finetune(config, seed, sets=(), label=None):
    """One src.vitonly.finetune run (this branch's ViT-only model) -> run_dir."""
    cmd = [sys.executable, "-m", "src.vitonly.finetune", "--config", config, "--seed", str(seed),
           "--cache-root", "/kaggle/input", "/kaggle/working", "--out", "/kaggle/working/runs"]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    print("\n" + " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d

def note_runs(key, dirs):
    """label -> run folder names, in OUT/runs.json: step 8 uses it to find the checkpoints."""
    RUNS_JSON[key] = [os.path.basename(d) for d in dirs]
    json.dump(RUNS_JSON, open(f"{OUT}/runs.json", "w"), indent=1)

def preds_of(dirs):
    fr = [ev.load_preds(f"{d}/val_preds.csv") for d in dirs]
    return fr[0] if len(fr) == 1 else ev.ensemble(fr)

def save_preds(label, df):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    df.to_csv(f"{OUT}/{label}/val_preds.csv", index=False)
    return df

def repo_preds(pattern):
    hits = sorted(glob.glob(f"results/*/{pattern}/val_preds.csv"))
    return preds_of([os.path.dirname(h) for h in hits]) if hits else None

def train_auc(d):
    """AUC on the training patients from export_preds.csv (B3 exports after training): the overfitting gap."""
    f = f"{d}/export_preds.csv"
    if not os.path.exists(f): return np.nan
    e = pd.read_csv(f); e = e[e.split == "train"]
    return stats.auc(e.label, e.prob)

def compare(preds, ref):
    """One row per model on the patients all models share: AUC (DeLong CI), AUPRC, Brier, ECE, the 90%-spec
    operating point (threshold chosen on these same val patients: optimistic), and the paired DeLong test
    and paired bootstrap of the AUC difference vs `ref`."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    rows = []
    for k, d in preds.items():
        r = {"model": k, **ev.summary_row(d.label, d.prob)}
        if k != ref and ref in preds:
            y = preds[ref].label
            dl = ev.delong(y, preds[ref].prob, d.prob)
            pb = stats.paired_bootstrap_auc_diff(y, preds[ref].prob, d.prob)
            r.update(diff_vs_ref=dl["diff"], delong_ci_lo=dl["ci"][0], delong_ci_hi=dl["ci"][1], delong_p=dl["p"],
                     boot_p=pb["p"])
        rows.append(r)
    cols = ["model", "n", "pcr", "auc", "ci_lo", "ci_hi", "auprc", "brier", "ece", "sens@90spec", "ppv@90spec",
            "npv@90spec", "diff_vs_ref", "delong_ci_lo", "delong_ci_hi", "delong_p", "boot_p"]
    t = pd.DataFrame(rows)
    return t[[c for c in cols if c in t]]

def subtype_wide(preds, by="subtype"):
    """AUC (95% CI) per subtype (or cohort) for every model, all on the same patients; n / pCR+ per group."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    tabs = {k: ev.subgroup_table(d, by).set_index(by) for k, d in preds.items()}
    first = next(iter(tabs.values()))
    out = pd.DataFrame({"n": first.n.astype(int), "pCR+": first.pcr.astype(int)})
    for k, t in tabs.items():
        out[k] = [f"{a:.3f} ({lo:.2f}-{hi:.2f})" if a == a else "n/a (one class)"
                  for a, lo, hi in zip(t.auc, t.ci_lo, t.ci_hi)]
    return out

B1_VAL = repo_preds("b1_lr_s0_*")                       # clinical + tumour size, logistic regression (val 0.653)
B0_VAL = repo_preds("b0_lr_s0_*")                       # clinical only (val 0.639)
AKULA_B3_V2 = repo_preds("b3_vit_s0_1019-0302_*")       # Akula's B3 on cache v2 (val 0.559)
B3_V1 = repo_preds("b3_vit_s0_1009-1616_*")             # B3 on the old cache (val 0.530)
F2_V1 = repo_preds("f2_attn_s0_1009-1605_*")            # fusion on the old cache (val 0.690)
print("reference predictions from the repo:", {k: v is not None for k, v in
      dict(B1=B1_VAL, B0=B0_VAL, B3_v2_Akula=AKULA_B3_V2, B3_v1=B3_V1, F2_v1=F2_V1).items()})

In [ ]:
caches = check_caches(CACHE_VERSION)

## 1. B3 reproduction (Akula's procedure)

In [ ]:
preds = {"B3, old cache v1 (repo)": B3_V1, "B3, cache v2 (Akula, repo)": AKULA_B3_V2}
if RUN_B3:
    d, m = train("b3_vit", 0, [], label="step6_b3_repro")
    note_runs("b3_repro", [d])
    preds["B3, cache v2 (reproduced)"] = ev.load_preds(f"{d}/val_preds.csv")
    a, b = ev.align({"a": AKULA_B3_V2, "b": preds["B3, cache v2 (reproduced)"]}).values()
    print(f"\nAkula val AUC {stats.auc(a.label, a.prob):.3f} | reproduced {stats.auc(b.label, b.prob):.3f} | "
          f"correlation of the two runs' predictions r = {np.corrcoef(a.prob, b.prob)[0, 1]:.2f} | "
          f"train AUC (overfitting) {train_auc(d):.3f} (Akula: 0.873)")
display(compare(preds, "B3, cache v2 (Akula, repo)").round(3))

## 2. Probe on cache v2 (frozen backbones, CV AUC on train)

In [ ]:
if RUN_PROBE:
    pdir = "/kaggle/working/probe_v2"
    cmd = [sys.executable, "-m", "src.vitonly.probe", "--backbones", *PROBE_BACKBONES, "--norms", *PROBE_NORMS,
           "--crops", *PROBE_CROPS, "--prefer", "uint8", "--out", pdir, "--cache-root", "/kaggle/input", "/kaggle/working"]
    if DEBUG: cmd += ["--limit", "40"]
    print(" ".join(cmd)); subprocess.run(cmd, check=True)
    probe = pd.read_csv(f"{pdir}/probe_results.csv")
    os.makedirs(f"{OUT}/step6_probe", exist_ok=True); probe.to_csv(f"{OUT}/step6_probe/probe_results.csv", index=False)
    display(probe.head(15).round(3))
    top = probe[probe.backbone != "-"].iloc[0]
    TOP = dict(backbone=top.backbone, norm=top.norm, crop=top.crop, pool=top["pool"])
else:
    TOP = dict(backbone="dinov2_b", norm="sub", crop="tumor", pool="gap")     # step-4 winner on the old cache
print("TOP setting for fine-tuning:", TOP)

## 3. Our ViT-only model on cache v2 (3 seeds + ensemble) vs B3

In [ ]:
if RUN_VITONLY:
    cfg = "vo_dinov2" if TOP["backbone"].startswith("dinov2") else "vo_base"
    sets = [f"data.norm={TOP['norm']}", f"data.crop={TOP['crop']}", f"model.backbone={TOP['backbone']}",
            f"model.pool={TOP['pool']}", "data.prefer_cache=uint8", *VITONLY_EXTRA]
    dirs = []
    for s in VITONLY_SEEDS:
        if budget_left() < 0.7:
            print(f"seed {s} skipped: time budget"); continue
        dirs.append(finetune(cfg, s, sets, f"step6_vitonly_s{s}"))
    note_runs("vitonly", dirs)
    preds["ViT-only vaib, cache v2 (ensemble)"] = save_preds("step6_vitonly_ensemble", preds_of(dirs))
    for d in dirs:
        print(os.path.basename(d), "val AUC", round(json.load(open(f"{d}/metrics.json"))["val"]["auc"], 3))
tab = compare(preds, "B3, cache v2 (Akula, repo)")
tab.to_csv(f"{OUT}/step6_summary.csv", index=False)
display(tab.round(3))
print("diff_vs_ref > 0 with delong_p < 0.05 = significantly better than Akula's B3 on the same 177 patients")

In [ ]:
# ==== PUSH the small result files to the vaib branch (results/<PERSON>/...) ====
if PUSH_RESULTS and not _tok:
    print("PUSH_RESULTS needs the GITHUB_TOKEN secret; download results/ from the Output tab instead")
elif PUSH_RESULTS:
    git("config", "user.name", f"Person {PERSON} (Kaggle)")
    git("config", "user.email", "kaggle@users.noreply.github.com")
    for attempt in range(3):
        git("fetch", "--quiet", "--depth", "1", "origin", f"+refs/heads/{GIT_TAG}:refs/remotes/origin/{GIT_TAG}")
        git("checkout", "--quiet", "-f", "-B", GIT_TAG, f"origin/{GIT_TAG}")
        shutil.copytree(f"/kaggle/working/results/{PERSON}", f"results/{PERSON}", dirs_exist_ok=True)
        git("add", f"results/{PERSON}")
        print(git("commit", "-m", f"results: {PERSON} {NOTEBOOK_NAME} ({GIT_TAG})"))
        try:
            git("push", "--quiet", "origin", GIT_TAG); print("pushed to", GIT_TAG); break
        except RuntimeError as e:
            print(f"push attempt {attempt + 1} failed:", e)
    else:
        print("NOT pushed: download results/ from the Output tab")